# AvatarGAN

A conditional GAN for generating cartoon avatars conditioned on discrete facial attributes (facial hair, hair style, face color, hair color).


## Imports and Configuration

Import all required libraries and define global hyperparameters: image size, latent space dimensionality, embedding size, and the path to the dataset.


In [ ]:
import os
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
image_size = 128
latent_dim = 128
embedding_dim = 64
img_shape = (3, image_size, image_size)
root_dir = "../cartoonset10k"

print(f"Using device: {device}")


## Dataset

`CustomImageDataset` loads PNG images from the CartoonSet directory and pairs each image with its attribute vector read from the corresponding CSV file. Only 4 attributes are used: `facial_hair`, `hair`, `face_color`, and `hair_color`.


In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, attr_names=None, max_variants=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = [
            os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith('.png')
        ]
        self.attr_names = attr_names
        self.max_variants = max_variants

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        csv_path = img_path.replace('.png', '.csv')
        attributes = self._load_csv_attributes(csv_path)
        attributes_tensor = torch.tensor(attributes, dtype=torch.long)

        return image, attributes_tensor

    def _load_csv_attributes(self, csv_path):
        attributes = []
        target_line = 12  # Line number to read (numbering from 1) - just an experiment to limit attributes
        with open(csv_path, 'r') as f:
            for i, line in enumerate(f, start=1):
                if i == target_line or i == target_line + 1 or i == target_line - 2 or i == target_line - 3:
                    parts = line.strip().split(',')
                    attr_name = parts[0].strip('"')
                    variant, total_variants = map(int, parts[1:])
                    attributes.append(variant)
        return attributes


## Model Architecture

- **AttributeBlock** — a small two-layer MLP that processes each attribute embedding independently before it is fed to the Generator/Discriminator.
- **Generator** — takes a latent noise vector `z` and per-attribute embeddings, processes them through fully-connected layers, and outputs a 128×128 RGB image.
- **Discriminator** — takes a flat image and per-attribute embeddings and outputs a real/fake probability score.


In [ ]:
class AttributeBlock(nn.Module):
    def __init__(self, embedding_dim):
        super(AttributeBlock, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 128),  # Reducing number of neurons
            nn.ReLU(),
            nn.Linear(128, embedding_dim),
            nn.ReLU()
        )

    def forward(self, attr_embed):
        return self.fc(attr_embed)


class Generator(nn.Module):
    def __init__(self, max_variants, embedding_dim, latent_dim):
        super(Generator, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(num_embeddings, embedding_dim) for num_embeddings in max_variants])
        self.attribute_blocks = nn.ModuleList([AttributeBlock(embedding_dim) for _ in max_variants])
        self.latent_fc = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(512 + len(max_variants) * embedding_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, int(torch.prod(torch.tensor(img_shape)))),
            nn.Tanh()
        )

    def forward(self, z, attrs):
        attr_embeddings = [embedding(attrs[:, i]) for i, embedding in enumerate(self.embeddings)]
        attr_embeddings = [block(attr_embed) for attr_embed, block in zip(attr_embeddings, self.attribute_blocks)]
        attr_embed = torch.cat(attr_embeddings, dim=1)
        z = self.latent_fc(z)
        combined = torch.cat((z, attr_embed), dim=1)
        img = self.fc(combined)
        return img.view(img.size(0), *img_shape)


class Discriminator(nn.Module):
    def __init__(self, max_variants, embedding_dim):
        super(Discriminator, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(num_embeddings, embedding_dim) for num_embeddings in max_variants])
        self.fc = nn.Sequential(
            nn.Linear(int(torch.prod(torch.tensor(img_shape))) + len(max_variants) * embedding_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img, attrs):
        img_flat = img.view(img.size(0), -1)
        attr_embeddings = [F.normalize(embedding(attrs[:, i]), p=2, dim=1) for i, embedding in enumerate(self.embeddings)]
        attrs_embed = torch.cat(attr_embeddings, dim=1)
        x = torch.cat((img_flat, attrs_embed), dim=1)
        return self.fc(x)


## Dataset and Model Setup

Load the dataset with augmentation transforms, create the dataloader, instantiate Generator and Discriminator, apply weight initialization, and define the optimizers and loss function.


In [ ]:
# Dataset and model setup
attr_names = ["facial_hair", "hair", "face_color", "hair_color"]
max_variants = [15, 111, 11, 10]

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

dataset = CustomImageDataset("../cartoonset10k", transform=transform, attr_names=attr_names, max_variants=max_variants)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Initialize models
generator = Generator(max_variants, embedding_dim, latent_dim).to(device)
discriminator = Discriminator(max_variants, embedding_dim).to(device)

# Weight initialization
generator.apply(lambda m: nn.init.normal_(m.weight, 0, 0.02) if hasattr(m, 'weight') else None)
discriminator.apply(lambda m: nn.init.normal_(m.weight, 0, 0.02) if hasattr(m, 'weight') else None)

# Optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999), weight_decay=1e-4)
# optimizer_G = optim.Adam(generator.parameters(), lr=0.0005, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))

adversarial_loss = nn.BCELoss()


## Helper Functions

- `get_random_attributes_from_csv` — samples a random subset of images and returns their attribute tensors and filenames.
- `save_model` — saves a full training checkpoint every 20 epochs.
- A fixed set of 6 samples is prepared here to allow consistent visual tracking of generation quality throughout training.


In [ ]:
def get_random_attributes_from_csv(dataset, num_samples=6):
    random_indices = random.sample(range(len(dataset)), num_samples)
    attributes = []
    filenames = []
    for idx in random_indices:
        img_path = dataset.image_paths[idx]
        filenames.append(os.path.basename(img_path))
        attributes.append(dataset[idx][1])
    return torch.stack(attributes), filenames


def save_model(generator, discriminator, optimizer_G, optimizer_D, epoch, save_dir="models4"):
    if epoch % 20 == 0 and epoch != 0:
        os.makedirs(save_dir, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
        }, os.path.join(save_dir, f"gan_epoch_{epoch}.pth"))
        print(f"Model saved at epoch {epoch} to {save_dir}/gan_epoch_{epoch}.pth")


# Sample a fixed batch of images and attributes for consistent visual evaluation during training
fixed_idxs = torch.randint(0, len(dataset), (6,))
fixed_samples = [dataset[i] for i in fixed_idxs]
fixed_attrs, fixed_filenames = get_random_attributes_from_csv(dataset, num_samples=6)
fixed_attrs = fixed_attrs.to(device)


## Training

The training loop runs for 360 epochs. Each epoch updates the Discriminator and Generator in turn. Every epoch a visual comparison of fixed original images and generated counterparts is displayed.


In [ ]:
# Model training
for epoch in range(360):
    for imgs, attrs in dataloader:
        real_imgs, attrs = imgs.to(device), attrs.to(device)
        valid = torch.ones((imgs.size(0), 1), device=device)
        fake = torch.zeros((imgs.size(0), 1), device=device)

        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs, attrs), valid)

        z = torch.randn((imgs.size(0), latent_dim), device=device)
        gen_imgs = generator(z, attrs)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach(), attrs), fake)
        d_loss = real_loss + fake_loss
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        g_loss = adversarial_loss(discriminator(gen_imgs, attrs), valid)
        g_loss.backward()
        optimizer_G.step()

    print(f"Epoch {epoch + 1}: [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")

    if epoch % 1 == 0:
        with torch.no_grad():
            z = torch.randn((6, latent_dim), device=device)
            gen_imgs = generator(z, fixed_attrs).cpu()

            fig, axes = plt.subplots(2, 6, figsize=(18, 8))
            for i in range(6):
                # Original images
                original_img_path = os.path.join(root_dir, fixed_filenames[i])
                original_img = Image.open(original_img_path).convert("RGB")
                original_img = transforms.ToTensor()(original_img)
                axes[0, i].imshow(original_img.permute(1, 2, 0) * 0.5 + 0.5)
                axes[0, i].axis("off")
                axes[0, i].set_title(f"Original: {fixed_filenames[i]}")

                # Generated images
                axes[1, i].imshow(gen_imgs[i].permute(1, 2, 0) * 0.5 + 0.5)
                axes[1, i].axis("off")
                axes[1, i].set_title("Generated")

            plt.tight_layout()
            plt.show()

    save_model(generator, discriminator, optimizer_G, optimizer_D, epoch)
